In [ ]:
import numpy as np
import pandas as pd
import os
import ruptures as rpt
import multiprocessing
from concurrent import futures
from tqdm import tqdm
import sys


from pyS3M import IOFunctions
from pyS3M import SpectralFunctions
from pyS3M import HelperFunctions
from IPython.display import display

IO = IOFunctions.IO_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
H_F = HelperFunctions.Helper_Functions()

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])

In [ ]:
# --- Configuration -----------------------------------------------------------
dye_data_folder = '/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/Brendan/20260202_HollidayJunctions/200mMMgCl2/20pM_HJ_20p561_568lp_DC2_emptyfilterweheel_100msExposure_1/'

# Change point detection parameters
CP_MODEL    = "l2"   # cost model for ruptures
CP_MIN_SIZE = 25     # minimum segment size (frames)
CP_JUMP     = 1      # grid of CP candidates

# Initial-guess filter: remove rows where A_R, A_G, A_B are all ~0.33
INITIAL_GUESS_TOL = 0.01

# Set True to skip files whose output already exists (useful for resuming)
SKIP_EXISTING = True
# -----------------------------------------------------------------------------

In [ ]:
all_h5_files = H_F.file_search(dye_data_folder, ".h5", "")

# Exclude any previously-saved changepoint files so we don't re-process them
loc_files = [f for f in all_h5_files if not f.endswith('_colourchangepoint.h5')]

print(f"Found {len(loc_files)} analysis H5 files to process")
for i, f in enumerate(loc_files):
    out_path = os.path.splitext(f)[0] + '_colourchangepoint.h5'
    status = "EXISTS" if os.path.exists(out_path) else "pending"
    print(f"  [{i:02d}] {os.path.basename(f)}  [{status}]")

In [ ]:
# Spectral setup — same filters as DNA_HJ_PostHoc_Changepoints.ipynb
dichroic_mirror  = "semrock-di03-r488-561-t1-25x36"
longpass_filter  = "semrock-blp01-568r"
filters          = [dichroic_mirror, longpass_filter]

dye = "Cy3"
_, dye_pixel_efficiency = S_F.get_pixel_fractions_dye_and_filters(dye, filters, wavelength, pixel_QYs)
dye_pixel_efficiency_Cy3 = dye_pixel_efficiency / np.sum(dye_pixel_efficiency)

dye = "Cy5"
_, dye_pixel_efficiency = S_F.get_pixel_fractions_dye_and_filters(dye, filters, wavelength, pixel_QYs)
dye_pixel_efficiency_Cy5 = dye_pixel_efficiency / np.sum(dye_pixel_efficiency)

print(f"Cy3 pixel efficiencies (B, G, R): {dye_pixel_efficiency_Cy3}")
print(f"Cy5 pixel efficiencies (B, G, R): {dye_pixel_efficiency_Cy5}")

In [ ]:
def filter_initial_guess(df, tol=0.01):
    """Remove rows where A_R, A_G, A_B are all within *tol* of 0.33.

    These represent the initial guess that was never updated by the fitter.

    Args:
        df (pd.DataFrame): DataFrame with A_R, A_G, A_B columns.
        tol (float): Tolerance for comparison to 0.33.

    Returns:
        pd.DataFrame: Copy with initial-guess rows removed.
    """
    mask = (
        np.isclose(df['A_R'], 0.33, atol=tol) &
        np.isclose(df['A_G'], 0.33, atol=tol) &
        np.isclose(df['A_B'], 0.33, atol=tol)
    )
    n_removed = mask.sum()
    print(f"  Removed {n_removed}/{len(df)} initial-guess rows ({100*n_removed/len(df):.1f}%)")
    return df[~mask].copy()


def find_colour_CPs(A_R, A_G, model="l2", min_size=25, jump=1):
    """Find change points jointly in A_R and A_G using multivariate detection.

    Uses BIC penalty (log(n) * dim * sigma^2).  A_R and A_G are anti-correlated
    spectral fractions so a FRET transition shifts both simultaneously; joint
    detection captures this correlated change.

    Args:
        A_R (np.ndarray): Red spectral fraction time series.
        A_G (np.ndarray): Green spectral fraction time series.
        model (str): Cost model for ruptures.
        min_size (int): Minimum segment size.
        jump (int): Grid of change-point candidates.

    Returns:
        list: Change-point indices; last element is always len(signal).
    """
    n = len(A_R)
    if n < min_size:
        return [n]
    signal = np.column_stack([A_R, A_G])
    dim = signal.shape[1]
    sigma2 = np.nanmean([np.nanvar(A_R), np.nanvar(A_G)])
    if sigma2 == 0:
        return [n]
    pen = np.log(n) * dim * sigma2
    algo = rpt.Pelt(model=model, min_size=min_size, jump=jump).fit(signal)
    return algo.predict(pen=pen)


def _find_CPs_for_punctum(args):
    """Worker: find change points for one punctum (used by ProcessPoolExecutor).

    Args:
        args (tuple): (puncta_id, A_R_values, A_G_values)

    Returns:
        tuple: (puncta_id, CPs, has_changepoint)
    """
    puncta_id, A_R, A_G = args
    CPs = find_colour_CPs(A_R, A_G)
    has_cp = len(CPs) > 1  # ruptures always appends the sentinel final index
    return (puncta_id, CPs, has_cp)


def find_CPs_parallel(df):
    """Run joint change-point detection on all puncta in parallel.

    Args:
        df (pd.DataFrame): Filtered DataFrame with puncta_id, A_R, A_G, frame.

    Returns:
        dict: {puncta_id: CPs} for puncta with at least one change point.
    """
    puncta_ids = df['puncta_id'].unique()
    tasks = []
    for pid in puncta_ids:
        sub = df[df['puncta_id'] == pid].sort_values('frame')
        tasks.append((pid, sub['A_R'].values.astype(np.float64),
                           sub['A_G'].values.astype(np.float64)))

    n_workers = min(60, max(1, int(0.9 * multiprocessing.cpu_count())))
    cp_results = {}

    with futures.ProcessPoolExecutor(n_workers) as executor:
        fs = {executor.submit(_find_CPs_for_punctum, t): t[0] for t in tasks}
        with tqdm(desc="  Finding colour CPs", total=len(tasks), unit="punctum") as pbar:
            for f in futures.as_completed(fs):
                pbar.update()
                puncta_id, CPs, has_cp = f.result()
                if has_cp:
                    cp_results[puncta_id] = CPs

    n_total = len(puncta_ids)
    n_cp = len(cp_results)
    print(f"  Puncta with change points: {n_cp}/{n_total} ({100*n_cp/n_total:.1f}%)")
    return cp_results


def compute_segment_stats(df_filtered, cp_results):
    """Compute per-segment summary statistics for all puncta with change points.

    For each segment between consecutive change points, computes the mean and
    standard deviation of A_R, A_G, A_B, the R/G ratio, and photons.  Frame
    boundaries are stored as actual frame numbers (not positional indices) so
    the table can be used directly for plotting without re-loading df_filtered.

    Args:
        df_filtered (pd.DataFrame): Filtered localisation data with columns
            puncta_id, frame, A_R, A_G, A_B, photons.
        cp_results (dict): {puncta_id: [cp_indices]} from find_CPs_parallel.
            The trailing sentinel (= len(signal)) is dropped internally.

    Returns:
        pd.DataFrame: One row per segment with columns:
            puncta_id, segment_number, frame_start, frame_end, n_points,
            A_R_mean, A_R_std, A_G_mean, A_G_std, A_B_mean, A_B_std,
            RG_ratio_mean, RG_ratio_std, photons_mean, photons_std.
    """
    _COLS = [
        'puncta_id', 'segment_number', 'frame_start', 'frame_end', 'n_points',
        'A_R_mean', 'A_R_std', 'A_G_mean', 'A_G_std', 'A_B_mean', 'A_B_std',
        'RG_ratio_mean', 'RG_ratio_std', 'photons_mean', 'photons_std',
    ]
    rows = []
    for pid, cps in cp_results.items():
        sub = (df_filtered[df_filtered['puncta_id'] == pid]
               .sort_values('frame')
               .reset_index(drop=True))
        real_cps = cps[:-1]                         # drop ruptures sentinel
        boundaries = [0] + list(real_cps) + [len(sub)]
        for seg_num, (s, e) in enumerate(zip(boundaries[:-1], boundaries[1:]), start=1):
            seg = sub.iloc[s:e]
            if len(seg) == 0:
                continue
            rg = seg['A_R'] / seg['A_G'].replace(0, np.nan)
            rows.append({
                'puncta_id':      pid,
                'segment_number': seg_num,
                'frame_start':    int(seg['frame'].min()),
                'frame_end':      int(seg['frame'].max()),
                'n_points':       len(seg),
                'A_R_mean':       seg['A_R'].mean(),
                'A_R_std':        seg['A_R'].std(),
                'A_G_mean':       seg['A_G'].mean(),
                'A_G_std':        seg['A_G'].std(),
                'A_B_mean':       seg['A_B'].mean(),
                'A_B_std':        seg['A_B'].std(),
                'RG_ratio_mean':  rg.mean(),
                'RG_ratio_std':   rg.std(),
                'photons_mean':   seg['photons'].mean(),
                'photons_std':    seg['photons'].std(),
            })
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=_COLS)


In [ ]:
def save_changepoint_results(out_path, df_filtered, cp_results):
    """Save change-point analysis to an HDF5 file.

    Three keys are written:

    'data'
        Filtered localisation DataFrame (initial guesses removed).  Contains
        all original columns (xc, yc, A_R, A_G, A_B, photons, frame, …).

    'changepoints'
        Long-format DataFrame, one row per change point:
            puncta_id  — punctum identifier
            cp_index   — positional index into the frame-sorted subset
            cp_number  — 1-based counter within that punctum
            frame      — actual frame number at the change point

    'segments'
        One row per segment between change points (see compute_segment_stats):
            puncta_id, segment_number, frame_start, frame_end, n_points,
            A_R_mean/std, A_G_mean/std, A_B_mean/std,
            RG_ratio_mean/std, photons_mean/std

    The ruptures sentinel (always = len(signal)) is *not* stored anywhere.

    Args:
        out_path (str): Destination .h5 file path.
        df_filtered (pd.DataFrame): Filtered localisation data.
        cp_results (dict): {puncta_id: [cp_indices]} from find_CPs_parallel.

    Returns:
        str: out_path (for confirmation).
    """
    # --- data ---
    df_filtered.to_hdf(out_path, key='data', mode='w', complevel=5, complib='blosc')

    # --- change-points (with actual frame numbers) ---
    cp_rows = []
    for pid, cps in cp_results.items():
        sub = (df_filtered[df_filtered['puncta_id'] == pid]
               .sort_values('frame')
               .reset_index(drop=True))
        real_cps = cps[:-1]
        for i, cp_idx in enumerate(real_cps):
            frame_val = int(sub['frame'].iloc[cp_idx]) if cp_idx < len(sub) else None
            cp_rows.append({
                'puncta_id': pid,
                'cp_index':  int(cp_idx),
                'cp_number': i + 1,
                'frame':     frame_val,
            })
    if cp_rows:
        cp_df = pd.DataFrame(cp_rows)
    else:
        cp_df = pd.DataFrame(columns=['puncta_id', 'cp_index', 'cp_number', 'frame'])
    cp_df.to_hdf(out_path, key='changepoints', mode='a', complevel=5, complib='blosc')

    # --- segment statistics ---
    seg_df = compute_segment_stats(df_filtered, cp_results)
    seg_df.to_hdf(out_path, key='segments', mode='a', complevel=5, complib='blosc')

    return out_path


def load_changepoint_analysis(changepoint_h5_path):
    """Load all tables from a _colourchangepoint.h5 file.

    Convenience reader for the plotting notebook.  Returns the three tables
    written by save_changepoint_results() so you can immediately plot traces,
    segment means, or aggregate statistics without recomputing anything.

    Usage example::

        df_data, cp_df, seg_df = load_changepoint_analysis(path)

        # plot segment means for one punctum
        pid = 42
        sub  = df_data[df_data['puncta_id'] == pid].sort_values('frame')
        segs = seg_df[seg_df['puncta_id'] == pid]
        cps  = cp_df[cp_df['puncta_id'] == pid]

        t = sub['frame'].values * exposure_time
        plt.scatter(t, sub['A_R'], s=4, c='red')
        for _, row in segs.iterrows():
            plt.hlines(row['A_R_mean'],
                       row['frame_start'] * exposure_time,
                       row['frame_end']   * exposure_time,
                       colors='darkred', lw=2)
        for _, cp in cps.iterrows():
            plt.axvline(cp['frame'] * exposure_time, ls='--', color='k')

    Args:
        changepoint_h5_path (str): Path to a *_colourchangepoint.h5 file.

    Returns:
        tuple:
            df_data (pd.DataFrame): Filtered localisation data.
            cp_df   (pd.DataFrame): Change-points table.
            seg_df  (pd.DataFrame): Segment-statistics table.
    """
    df_data = pd.read_hdf(changepoint_h5_path, key='data')
    cp_df   = pd.read_hdf(changepoint_h5_path, key='changepoints')
    seg_df  = pd.read_hdf(changepoint_h5_path, key='segments')
    return df_data, cp_df, seg_df


In [ ]:
n_files = len(loc_files)
print(f"Processing {n_files} files...\n")

summary_rows = []

for file_idx, h5_path in enumerate(loc_files):
    fname    = os.path.basename(h5_path)
    out_path = os.path.splitext(h5_path)[0] + '_colourchangepoint.h5'

    print(f"[{file_idx+1}/{n_files}] {fname}")

    if SKIP_EXISTING and os.path.exists(out_path):
        print(f"  SKIP — output already exists: {os.path.basename(out_path)}\n")
        summary_rows.append({'file': fname, 'status': 'skipped',
                             'n_puncta': None, 'n_cp_puncta': None})
        continue

    try:
        # 1. load
        df = pd.read_hdf(h5_path)
        n_puncta_total = df['puncta_id'].nunique()
        print(f"  Loaded {len(df)} rows, {n_puncta_total} unique puncta")

        # 2. filter initial guesses
        df_filtered = filter_initial_guess(df, tol=INITIAL_GUESS_TOL)

        # 3. find change points (parallel)
        cp_results = find_CPs_parallel(df_filtered)

        # 4. save
        save_changepoint_results(out_path, df_filtered, cp_results)

        n_cp = len(cp_results)
        print(f"  Saved -> {os.path.basename(out_path)}\n")
        summary_rows.append({'file': fname, 'status': 'ok',
                             'n_puncta': n_puncta_total, 'n_cp_puncta': n_cp})

    except Exception as e:
        print(f"  ERROR: {e}\n")
        summary_rows.append({'file': fname, 'status': f'error: {e}',
                             'n_puncta': None, 'n_cp_puncta': None})

# --- Summary -----------------------------------------------------------------
print("\n" + "="*70)
print("Batch processing summary")
print("="*70)
summary_df = pd.DataFrame(summary_rows)
display(summary_df)